[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](<https://colab.research.google.com/github/MMartinelli-hub/ATA2026_Tutorship/blob/main/bert_overview/BERT_Overview.ipynb>)


# BERT Overview

### What you'll learn
- How tokenization works with BERT (wordpiece, padding strategies, and the **attention mask**).
- How to inspect BERT: configuration, architecture, vocabulary, special tokens, and **[unused]** tokens.
- How to use BERT for its original tasks: Masked Language Modelling and Next Sentence Prediction

## 1) Setup

In [ ]:
# If transformers/torch aren't installed in your environment, uncomment and run:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install transformers>=4.41.0 datasets==2.19.0

import os
import math
import random
import json
from dataclasses import dataclass
from typing import List, Dict, Tuple
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd

from transformers import BertTokenizerFast, BertConfig, BertModel, AutoConfig
from transformers.optimization import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2) Tokenizer: effects, padding, attention mask

In [ ]:
# Load the tokenizer
MODEL_NAME = "bert-base-uncased"  
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
# Inspect vocab & special tokens
print("Max length:", tokenizer.model_max_length)
print("Vocab size:", tokenizer.vocab_size)
print("Special tokens:", tokenizer.all_special_tokens)
for token_id in tokenizer.all_special_ids:
    print(tokenizer.decode(token_id), "->", token_id)

In [ ]:
# Show a few [unused] tokens (BERT ships many)
unused = [t for t in tokenizer.get_vocab().keys() if t.startswith("[unused")]
unused_digits = [int(t.replace("unused", "").replace("[", "").replace("]", "")) for t in unused]
unused_digits = sorted(unused_digits)
unused = [f"[unused{i}]" for i in unused_digits]
print("First 10 [unused] tokens:", unused[:10])
print("Last 10 [unused] tokens:", unused[-10:])

In [ ]:
#sample = "I absolutely loved the cinematography, but the acting was so-so."
sample = "Tokenization and padding are fundamental steps in most NLP pipelines."
enc = tokenizer(sample, return_tensors="pt")

print("Original text:", sample)
tokens = []
for token_id in enc["input_ids"][0].tolist():
    tokens.append(tokenizer.decode(token_id))
print("Tokenized:", tokens)
print("Num of tokens:", len(tokens))

print("\n")
for k,v in enc.items():
    print(k, v.shape, v[0])

tokens = []
for token_id, attention_mask_id in zip(enc["input_ids"][0], enc["attention_mask"][0]):
    tokens.append({
        "token": tokenizer.decode(token_id),
        "token_id": token_id,
        "attention_mask_id": attention_mask_id
    })

print("\n\n")
print(pd.DataFrame.from_dict(tokens).to_markdown())

In [ ]:
import pandas as pd

# Padding strategies and attention mask
batch = [
    "This movie was amazing!",
    "Bad.",
    "I would not watch it again, honestly.",
]


try:
    enc_nopad = tokenizer(batch, padding=False, truncation=True, return_tensors="pt")
except Exception as e:
    print("Encoding strings with different lengths without padding gives error!")
    print("Error:", e)

print("\n--- Padding to a fixed `max_length` (e.g., 16) ---")
print("Purpose: Ensures all sequences have the exact same length, often used for fixed-size model inputs.")
enc_pad_left = tokenizer(batch, padding="max_length", max_length=16, truncation=True, return_tensors="pt") # if we don't specify the value for max length, it will take by default the one of the model
print("Input IDs shape:", enc_pad_left["input_ids"].shape)
print("Attention Mask shape:", enc_pad_left["attention_mask"].shape)
print("First sequence input_ids (padded):")
tokens = []
for token_id, attention_mask_id in zip(enc_pad_left["input_ids"][0], enc_pad_left["attention_mask"][0]):
    tokens.append({
        "token": tokenizer.decode(token_id),
        "token_id": token_id,
        "attention_mask_id": attention_mask_id
    })
print(pd.DataFrame.from_dict(tokens).to_markdown())

print("\n\n--- Padding to the `longest` sequence in the current batch ---")
print("Purpose: Minimizes padding by only matching the longest sequence in the batch, saving computation.")
enc_pad_longest = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
print("Input IDs shape:", enc_pad_longest["input_ids"].shape)
print("Attention Mask shape:", enc_pad_longest["attention_mask"].shape)
tokens = []
for token_id, attention_mask_id in zip(enc_pad_longest["input_ids"][2], enc_pad_longest["attention_mask"][2]):
    tokens.append({
        "token": tokenizer.decode(token_id),
        "token_id": token_id,
        "attention_mask_id": attention_mask_id
    })
print(pd.DataFrame.from_dict(tokens).to_markdown())


### Why attention masks matter

Let's pass two padded sequences through BERT **with** and **without** the attention mask and see how the outputs change.  
Padding tokens should **not** influence the representation of real tokens.


In [ ]:
config = BertConfig.from_pretrained(MODEL_NAME, output_hidden_states=True)
bert = BertModel.from_pretrained(MODEL_NAME, config=config).to(device)
bert.eval()

with torch.no_grad():
    padded = tokenizer(["hello world", "hello"], padding=True, return_tensors="pt").to(device)
    #padded = tokenizer(["hello world", "hello"], padding="max_length", max_length=50, return_tensors="pt").to(device)

    # With mask
    out_with = bert(input_ids=padded["input_ids"], attention_mask=padded["attention_mask"])
    # Without mask (pretend padding is real content)
    out_without = bert(input_ids=padded["input_ids"], attention_mask=None)

# Compare [CLS] embeddings difference
cls_with = out_with.last_hidden_state[:,0,:]
cls_without = out_without.last_hidden_state[:,0,:]
diff = (cls_with - cls_without).pow(2).sum(dim=-1).sqrt().cpu().tolist()
print("L2 distance between CLS with/without mask per sequence:", diff)


## 3) BERT architecture: modules, shapes, and parameters

In [ ]:
total_params = sum(p.numel() for p in bert.parameters())
trainable_params = sum(p.numel() for p in bert.parameters() if p.requires_grad)
print(f"Total params in {MODEL_NAME}: {total_params:,} (trainable: {trainable_params:,})")

print("\nHigh-level modules:")
for name, module in bert.named_children():
    print(" -", name, ":", module.__class__.__name__)

print("\nEncoder layer stack depth:", bert.config.num_hidden_layers)
print("Hidden size:", bert.config.hidden_size)
print("Intermediate FF size:", bert.config.intermediate_size)
print("Attention heads:", bert.config.num_attention_heads)


## 4) BERT's original pre-training tasks

BERT was originally pre-trained on two tasks:
1. **Masked Language Modeling (MLM)**: Predict masked tokens in a sentence.
2. **Next Sentence Prediction (NSP)**: Determine if sentence B follows sentence A.

Let's demonstrate both tasks using a pre-trained BERT model.

In [ ]:
# Masked Language Modeling (MLM) Demo
# We need BertForMaskedLM for this task
from transformers import AutoTokenizer, BertForMaskedLM
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

print(tokenizer.mask_token, "->", tokenizer.mask_token_id)
inputs = tokenizer("The capital of France is [MASK].", return_tensors="pt")

with torch.no_grad():
    # Get model predictions (logits) for all tokens in the vocabulary
    # logits shape: [batch_size=1, sequence_length, vocab_size=30522]
    logits = model(**inputs).logits

# Step 1: Find the position of [MASK] in the input sequence
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

# Step 2: Get the predicted token for the [MASK] position
# logits[0, mask_token_index] extracts predictions at the mask position (shape: [vocab_size])
# .argmax(axis=-1) finds the token ID with the highest score (most likely prediction)
predicted_token_id = logits[0, mask_token_index].argmax(axis=-1)

print("Predicted token id:", predicted_token_id.item(), "->", tokenizer.decode(predicted_token_id))  

# Step 3: Prepare labels for loss calculation
# Tokenize the correct sentence to get ground truth token IDs
labels = tokenizer("The capital of France is Paris.", return_tensors="pt")["input_ids"]

# Set all non-[MASK] positions to -100 (tells PyTorch to ignore them in loss calculation)
# Only the token at the [MASK] position will be used to compute the loss
# Result: [-100, -100, -100, -100, -100, -100, 3000, -100] where 3000 is "paris"
print("Tokens:", tokenizer.convert_ids_to_tokens(labels[0]))
print("Labels before masking non-[MASK]:", labels)
labels = torch.where(inputs.input_ids == tokenizer.mask_token_id, labels, -100)
print("Labels after masking non-[MASK]:", labels)

# Step 4: Calculate loss (how wrong was the prediction?)
# The model compares its prediction at [MASK] position with the correct token "Paris"
# Lower loss = better prediction (0 would mean perfect prediction)
outputs = model(**inputs, labels=labels)
print(f"Loss: {round(outputs.loss.item(), 2)}")  

### Understanding the Loss Value

Loss measures **confidence**, not just correctness!

- **Loss = -log(probability)** for the correct token
- Loss of 0.88 means BERT gave "paris" a probability of ~41% (exp(-0.88) ≈ 0.41)
- Even though "paris" was the top prediction, BERT wasn't 100% confident
- BERT must choose from **30,522 possible tokens** in the vocabulary!

**Loss interpretation:**
- Loss = 0.0 → 100% confident (probability = 1.0) - perfect!
- Loss = 0.69 → 50% confident (probability = 0.5)
- Loss = 1.0 → 37% confident (probability = 0.37)
- Loss = 2.0 → 14% confident (probability = 0.14)

Let's see the actual probabilities for the top predictions:

In [ ]:
# Let's look at the actual probabilities for top-10 predictions
import torch.nn.functional as F

# Get probabilities (not just logits)
probs_at_mask = F.softmax(logits[0, mask_token_index], dim=-1)

# Get top 10 predictions with their probabilities
top_k = 10
top_probs, top_indices = torch.topk(probs_at_mask, top_k, dim=-1)

print(f"Top {top_k} predictions for 'The capital of France is [MASK].':\n")
print(f"{'Rank':<6} {'Token':<15} {'Probability':<15} {'Loss if correct':<20}")
print("-" * 60)

for rank, (prob, idx) in enumerate(zip(top_probs[0], top_indices[0]), 1):
    token = tokenizer.decode([idx])
    prob_value = prob.item()
    loss_value = -torch.log(prob).item()
    
    # Highlight the actual prediction (paris)
    marker = " ← PREDICTED" if rank == 1 else ""
    print(f"{rank:<6} {token:<15} {prob_value:<15.4f} {loss_value:<20.4f}{marker}")

print("\n" + "="*60)
print(f"Notice: Even though 'paris' is #1, it has ~{top_probs[0][0].item()*100:.1f}% probability")
print(f"The model spreads probability across other plausible tokens.")
print(f"This is why the loss is {outputs.loss.item():.2f}, not 0.0!")

In [ ]:
# Next Sentence Prediction (NSP) Demo
# We need BertForNextSentencePrediction for this task
from transformers import BertForNextSentencePrediction

nsp_model = BertForNextSentencePrediction.from_pretrained(MODEL_NAME).to(device)
nsp_model.eval()

# Test sentences
starting_sentence = "The weather is beautiful today."
true_next = "I think I'll go for a walk in the park."
false_next = "Deep Learning is a branch of Machine Learning."

sentence_pairs = [
    (starting_sentence, true_next, "True continuation"),
    (starting_sentence, false_next, "False continuation"),
]

print("Next Sentence Prediction Demo\n")
print(f"Starting sentence: '{starting_sentence}'\n")

for sent_a, sent_b, description in sentence_pairs:
    # Tokenize the sentence pair
    # BERT uses [CLS] sent_a [SEP] sent_b [SEP] format
    inputs = tokenizer(sent_a, sent_b, return_tensors="pt").to(device)
    
    # Get predictions
    with torch.no_grad():
        outputs = nsp_model(**inputs)
        logits = outputs.logits
    
    # logits shape: [batch_size, 2]
    # logits[:, 0] = score for "is next sentence"
    # logits[:, 1] = score for "is NOT next sentence"
    probs = F.softmax(logits, dim=-1)
    is_next_prob = probs[0, 0].item()
    not_next_prob = probs[0, 1].item()
    
    prediction = "IS the next sentence" if is_next_prob > not_next_prob else "IS NOT the next sentence"
    
    print(f"--- {description} ---")
    print(f"Candidate sentence: '{sent_b}'")
    print(f"Prediction: {prediction}")
    print(f"  P(is next)     = {is_next_prob:.4f}")
    print(f"  P(is NOT next) = {not_next_prob:.4f}")
    print()


## 5) References
- Many other BertFor** are supported by transformers. You can check the full list at https://huggingface.co/docs/transformers/model_doc/bert
- Check out also the "Auto" classes, which allows for more flexible pipelines integrating BERT and non-BERT models without changing the code: https://huggingface.co/docs/transformers/model_doc/auto

## 6) Exercises 


### Exercise 1: Tokenization Analysis

**Task:** Tokenize the following sentence and analyze its components:
```
"The pre-trained BERT model uses WordPiece tokenization with 30,522 tokens."
```

**Requirements:**
1. Tokenize the sentence and print the tokens
2. Calculate the total number of tokens (including special tokens)
3. Identify which words got split into subwords
4. Create a DataFrame showing: token, token_id, and whether it's a subword (starts with ##)

**Bonus:** Try with a sentence containing a rare word like "antidisestablishmentarianism" and observe the tokenization.

### Exercise 2: Padding Strategy Comparison

**Task:** Compare different padding strategies.

Given the following batch of sentences:
```python
sentences = [
    "Great!",
    "This is a medium length sentence about machine learning.",
    "Short one.",
    "This is the longest sentence in our batch and it talks about natural language processing and transformers."
]
```

**Requirements:**
1. Tokenize with `padding="longest"` and report the shape
2. Tokenize with `padding="max_length"` (max_length=64) and report the shape
3. Calculate the percentage of padding tokens for each strategy
4. Discuss: Which strategy would be better for training? Why?

### Exercise 3: Exploring BERT's Architecture

**Task:** Investigate BERT's parameter distribution across different components.

**Requirements:**
1. Calculate the number of parameters in the embedding layer (word embeddings, position embeddings, token type embeddings)
2. Calculate the number of parameters in a single encoder layer
3. Calculate the number of parameters in the pooler layer
4. Verify your calculations sum to the total number of parameters

**Hint:** Use `named_parameters()` to iterate through all parameters and their shapes.

### Exercise 4: Masked Language Modeling with Multiple Masks

**Task:** Implement a function that predicts multiple masked tokens in a sentence and evaluates the predictions.

**Requirements:**
1. Create a function `predict_masked_tokens(sentence, mask_positions)` that:
   - Takes a sentence and a list of word positions to mask (0-indexed, excluding special tokens)
   - Replaces those positions with [MASK]
   - Returns the top-3 predictions for each masked position with their probabilities
2. Test your function with: `"The quick brown fox jumps over the lazy dog"` 
   - Mask positions: 1 ("quick"), 4 ("jumps"), and 7 ("lazy")
3. Calculate the average confidence (probability) of the top prediction across all masks
4. Discuss: Why might BERT struggle with certain masked words more than others?

### Exercise 5: Next Sentence Prediction Evaluation

**Task:** Create a small dataset to evaluate BERT's Next Sentence Prediction capabilities and analyze its performance.

**Requirements:**
Below are reported 10 sentence pairs from diverse topics (news, stories, technical content, etc.), 5 that are actual continuations and 5 that are not.
1. For each pair, get BERT's NSP prediction and confidence scores
2. Calculate accuracy, precision, and recall for the predictions
3. Identify which pair had the highest confidence (regardless of correctness)
4. Create a confusion matrix visualization

**Hint:** Use `BertForNextSentencePrediction` and apply softmax to get probabilities.

In [ ]:
# 1. Create dataset of sentence pairs
sentence_pairs = [
    # True continuations (label = 1 for "is next")
    {
        'sent_a': "The weather forecast predicts heavy rain tomorrow.",
        'sent_b': "Don't forget to bring an umbrella when you go out.",
        'is_next': True,
        'topic': 'Weather'
    },
    {
        'sent_a': "She studied hard for her final exams all week.",
        'sent_b': "Her efforts paid off when she received excellent grades.",
        'is_next': True,
        'topic': 'Education'
    },
    {
        'sent_a': "The company announced record profits in Q4.",
        'sent_b': "Shareholders were pleased with the financial results.",
        'is_next': True,
        'topic': 'Business'
    },
    {
        'sent_a': "Neural networks learn by adjusting their weights.",
        'sent_b': "This process is called backpropagation and uses gradient descent.",
        'is_next': True,
        'topic': 'Technical'
    },
    {
        'sent_a': "The movie had stunning visual effects and great acting.",
        'sent_b': "It went on to win several Academy Awards that year.",
        'is_next': True,
        'topic': 'Entertainment'
    },
    
    # False continuations (label = 0 for "is NOT next")
    {
        'sent_a': "Quantum computing uses qubits instead of classical bits.",
        'sent_b': "Pizza is one of the most popular foods in Italy.",
        'is_next': False,
        'topic': 'Random'
    },
    {
        'sent_a': "The patient showed symptoms of seasonal allergies.",
        'sent_b': "The stock market reached an all-time high today.",
        'is_next': False,
        'topic': 'Random'
    },
    {
        'sent_a': "Python is a versatile programming language.",
        'sent_b': "Pythons are the largest snakes in the world.",
        'is_next': False,
        'topic': 'Random'
    },
    {
        'sent_a': "Climate change affects global weather patterns.",
        'sent_b': "The restaurant serves authentic Japanese cuisine.",
        'is_next': False,
        'topic': 'Random'
    },
    {
        'sent_a': "The team celebrated their championship victory.",
        'sent_b': "Photosynthesis converts light energy into chemical energy.",
        'is_next': False,
        'topic': 'Random'
    }
]